# Simple Simulator Consistency Check

This notebook contains a simple function to verify:

**Does our simulator reproduce the sensor measurements in the CSV when given the same state and flight conditions?**

If YES -> Data is consistent, GA should work

If NO -> Data inconsistency, need to investigate why

In [27]:
import numpy as np
import pandas as pd

# Simulator imports
from odsmr.sensors import HPC_Tout, HP_Nmech, HPC_Tin, LPT_Tin, Fuel_flow, HPC_Pout_st, LP_Nmech
from odsmr.context import FlightCondDeckSMR, ContextDeckSMR
from odsmr.predefined_flight_conditions import Cruise_DeckSMR, Takeoff_DeckSMR, Climb1_DeckSMR, Climb2_DeckSMR
from odsmr.generation_functions import decksmr_1forall
from odsmr.constants import ROOT_OPENDECK, STATE_LABELS, STATE_BOUNDS

print("Imports OK")

Imports OK


In [35]:
STATE_LABELS

['deg_CmpBst_s_mapEff_in',
 'deg_CmpBst_s_mapWc_in',
 'deg_CmpFan_s_mapEff_in',
 'deg_CmpFan_s_mapWc_in',
 'deg_CmpH_s_mapEff_in',
 'deg_CmpH_s_mapWc_in',
 'deg_TrbH_s_mapEff_in',
 'deg_TrbH_s_mapWc_in',
 'deg_TrbL_s_mapEff_in',
 'deg_TrbL_s_mapWc_in']

In [ ]:
STATE_LABELS= ['deg_CmpBst_s_mapEff_in',
 'deg_CmpBst_s_mapWc_in',
 'deg_CmpFan_s_mapEff_in',
 'deg_CmpFan_s_mapWc_in',
 'deg_CmpH_s_mapEff_in',
 'deg_CmpH_s_mapWc_in',
 'deg_TrbH_s_mapEff_in',
 'deg_TrbH_s_mapWc_in',
 'deg_TrbL_s_mapEff_in',
 'deg_TrbL_s_mapWc_in']

In [28]:
# Configuration
DATA_FILE_PATH = "./data/Cie_0_Engine0.csv"

CONTEXT_TEMPLATE = {
    "CRUISE": Cruise_DeckSMR,
    "TAKEOFF": Takeoff_DeckSMR,
    "CLIMB1": Climb1_DeckSMR,
    "CLIMB2": Climb2_DeckSMR,
}

SENSOR_OBJECTS = {
    "HPC_Tout": HPC_Tout(), 
    "HP_Nmech": HP_Nmech(), 
    "HPC_Tin": HPC_Tin(),
    "LPT_Tin": LPT_Tin(), 
    "Fuel_flow": Fuel_flow(), 
    "HPC_Pout_st": HPC_Pout_st(),
    "LP_Nmech": LP_Nmech(),
}

# Load data
df = pd.read_csv(DATA_FILE_PATH)
print(f"Loaded {len(df)} rows")

Loaded 1000 rows


---
## Simple Check Function

In [32]:
def check_simulator_consistency(df, row_index, context_name, sensor_names):
    """
    Simple function to check if simulator reproduces CSV measurements.
    
    Args:
        df: DataFrame with CSV data
        row_index: Which row to test
        context_name: "CRUISE", "TAKEOFF", "CLIMB1", or "CLIMB2"
        sensor_names: List of sensor names, e.g. ["HPC_Tin", "LPT_Tin", "HPC_Pout_st"]
    
    Returns:
        dict with comparison results
    """
    row = df.iloc[row_index]
    prefix = f"{context_name}_"
    
    # =====================================================
    # STEP 1: Extract state vector (all 10 health indicators)
    # =====================================================
    state = np.array([row[label] for label in STATE_LABELS])

    print ("-----------------STATE---------------------\n")
    print (state)
    
    print(f"Row {row_index}, Context: {context_name}")
    print("="*60)
    print("\nSTEP 1: State vector from CSV")
    for i, label in enumerate(STATE_LABELS):
        print(f"  [{i}] {label}: {state[i]:.6f}")
    
    # =====================================================
    # STEP 2: Extract flight conditions from CSV
    # =====================================================
    fc_csv = {
        'DTAMB': row[prefix + "DTAMB"],
        'ALT': row[prefix + "ALT"],
        'MACH': row[prefix + "MACH"],
        'COMMAND': row[prefix + "COMMAND"],
    }
    
    print(f"\nSTEP 2: Flight conditions from CSV")
    for k, v in fc_csv.items():
        print(f"  {k}: {v}")
    
    # =====================================================
    # STEP 3: Create simulator context
    # =====================================================
    template = CONTEXT_TEMPLATE[context_name]
    
    flight_cond = FlightCondDeckSMR(
        PHASE_TYPE=template.flight_condition.PHASE_TYPE,
        DTAMB=float(fc_csv['DTAMB']),
        ALT=float(fc_csv['ALT']),
        MACH=float(fc_csv['MACH']),
        COMMAND=float(fc_csv['COMMAND']),
    )
    
    context = ContextDeckSMR(
        name=f"{context_name}_test",
        flight_condition=flight_cond
    )
    
    print(f"\nSTEP 3: Created context with PHASE_TYPE={flight_cond.PHASE_TYPE}")
    
    # =====================================================
    # STEP 4: Run simulator
    # =====================================================
    sensors_list = [SENSOR_OBJECTS[s] for s in sensor_names]
    
    result = decksmr_1forall([state], [context], sensors_list, ROOT_OPENDECK)
    
    print(f"\nSTEP 4: Simulator called successfully")
    
    # =====================================================
    # STEP 5: Compare with CSV values
    # =====================================================
    print(f"\nSTEP 5: COMPARISON")
    print(f"{'Sensor':<15} {'CSV Value':>15} {'Simulated':>15} {'Diff':>15} {'Match?':>10}")
    print("-"*70)
    
    results = {}
    all_match = True
    
    for sensor_name in sensor_names:
        csv_col = f"{context_name}_DECKSMR{sensor_name}"
        csv_value = row[csv_col]
        sim_value = result[sensor_name].values[0]
        diff = sim_value - csv_value
        rel_diff = 100 * abs(diff) / abs(csv_value) if csv_value != 0 else 0
        
        # Consider match if relative difference < 0.01%
        is_match = rel_diff < 0.01
        if not is_match:
            all_match = False
        
        match_str = "✓" if is_match else f"✗ ({rel_diff:.2f}%)"
        
        print(f"{sensor_name:<15} {csv_value:>15.4f} {sim_value:>15.4f} {diff:>15.4f} {match_str:>10}")
        
        results[sensor_name] = {
            'csv': csv_value,
            'simulated': sim_value,
            'diff': diff,
            'rel_diff_pct': rel_diff,
            'match': is_match,
        }
    
    # =====================================================
    # VERDICT
    # =====================================================
    print("\n" + "="*60)
    if all_match:
        print("✓ VERDICT: Simulator MATCHES CSV data!")
        print("  The data is consistent with our simulator.")
    else:
        print("✗ VERDICT: Simulator does NOT match CSV data!")
        print("  Possible reasons:")
        print("  1. CSV was generated with different simulator version")
        print("  2. Noise was added to CSV sensor values")
        print("  3. Wrong column mapping")
        print("  4. Different state vector ordering")
    print("="*60)
    
    return {
        'row_index': row_index,
        'context': context_name,
        'all_match': all_match,
        'sensors': results,
    }

In [33]:
STATE_LABELS

['deg_CmpBst_s_mapEff_in',
 'deg_CmpBst_s_mapWc_in',
 'deg_CmpFan_s_mapEff_in',
 'deg_CmpFan_s_mapWc_in',
 'deg_CmpH_s_mapEff_in',
 'deg_CmpH_s_mapWc_in',
 'deg_TrbH_s_mapEff_in',
 'deg_TrbH_s_mapWc_in',
 'deg_TrbL_s_mapEff_in',
 'deg_TrbL_s_mapWc_in']

---
## Run the Check

In [34]:
# Test with CRUISE context
result_cruise = check_simulator_consistency(
    df, 
    row_index=0, 
    context_name="CRUISE", 
    sensor_names=["HPC_Tin", "LPT_Tin", "HPC_Pout_st"]
)

-----------------STATE---------------------

[-6.11851027e-04 -1.74335760e-04  3.09300505e-05 -1.26495116e-05
 -1.94308813e-03  4.27521336e-05 -1.46328797e-03  6.49187104e-05
 -5.54972613e-04 -1.51935833e-05]
Row 0, Context: CRUISE

STEP 1: State vector from CSV
  [0] deg_CmpBst_s_mapEff_in: -0.000612
  [1] deg_CmpBst_s_mapWc_in: -0.000174
  [2] deg_CmpFan_s_mapEff_in: 0.000031
  [3] deg_CmpFan_s_mapWc_in: -0.000013
  [4] deg_CmpH_s_mapEff_in: -0.001943
  [5] deg_CmpH_s_mapWc_in: 0.000043
  [6] deg_TrbH_s_mapEff_in: -0.001463
  [7] deg_TrbH_s_mapWc_in: 0.000065
  [8] deg_TrbL_s_mapEff_in: -0.000555
  [9] deg_TrbL_s_mapWc_in: -0.000015

STEP 2: Flight conditions from CSV
  DTAMB: 9.422
  ALT: 35000.102656
  MACH: 0.780833
  COMMAND: 24961.358645

STEP 3: Created context with PHASE_TYPE=CR

STEP 4: Simulator called successfully

STEP 5: COMPARISON
Sensor                CSV Value       Simulated            Diff     Match?
-------------------------------------------------------------------

In [14]:
# Test with TAKEOFF context
print("\n\n")
result_takeoff = check_simulator_consistency(
    df, 
    row_index=0, 
    context_name="TAKEOFF", 
    sensor_names=["HPC_Tin", "LPT_Tin", "HPC_Pout_st"]
)




Row 0, Context: TAKEOFF

STEP 1: State vector from CSV
  [0] deg_CmpBst_s_mapEff_in: -0.000612
  [1] deg_CmpBst_s_mapWc_in: -0.000174
  [2] deg_CmpFan_s_mapEff_in: 0.000031
  [3] deg_CmpFan_s_mapWc_in: -0.000013
  [4] deg_CmpH_s_mapEff_in: -0.001943
  [5] deg_CmpH_s_mapWc_in: 0.000043
  [6] deg_TrbH_s_mapEff_in: -0.001463
  [7] deg_TrbH_s_mapWc_in: 0.000065
  [8] deg_TrbL_s_mapEff_in: -0.000555
  [9] deg_TrbL_s_mapWc_in: -0.000015

STEP 2: Flight conditions from CSV
  DTAMB: 15.023
  ALT: 76.922407
  MACH: 8.4e-05
  COMMAND: 24998.392463

STEP 3: Created context with PHASE_TYPE=MTO

STEP 4: Simulator called successfully

STEP 5: COMPARISON
Sensor                CSV Value       Simulated            Diff     Match?
----------------------------------------------------------------------
HPC_Tin                402.5946        386.4025        -16.1922  ✗ (4.02%)
LPT_Tin               1274.0902       1266.5044         -7.5858  ✗ (0.60%)
HPC_Pout_st        3404800.3600    3527355.8600     1

In [15]:
# Test with CLIMB1 context
print("\n\n")
result_climb1 = check_simulator_consistency(
    df, 
    row_index=0, 
    context_name="CLIMB1", 
    sensor_names=["HPC_Tin", "LPT_Tin", "HPC_Pout_st"]
)




Row 0, Context: CLIMB1

STEP 1: State vector from CSV
  [0] deg_CmpBst_s_mapEff_in: -0.000612
  [1] deg_CmpBst_s_mapWc_in: -0.000174
  [2] deg_CmpFan_s_mapEff_in: 0.000031
  [3] deg_CmpFan_s_mapWc_in: -0.000013
  [4] deg_CmpH_s_mapEff_in: -0.001943
  [5] deg_CmpH_s_mapWc_in: 0.000043
  [6] deg_TrbH_s_mapEff_in: -0.001463
  [7] deg_TrbH_s_mapWc_in: 0.000065
  [8] deg_TrbL_s_mapEff_in: -0.000555
  [9] deg_TrbL_s_mapWc_in: -0.000015

STEP 2: Flight conditions from CSV
  DTAMB: 9.954
  ALT: 14943.552335
  MACH: 0.544517
  COMMAND: 25019.886837

STEP 3: Created context with PHASE_TYPE=MCL

STEP 4: Simulator called successfully

STEP 5: COMPARISON
Sensor                CSV Value       Simulated            Diff     Match?
----------------------------------------------------------------------
HPC_Tin                374.9750        362.0518        -12.9232  ✗ (3.45%)
LPT_Tin               1147.0758       1153.7544          6.6787  ✗ (0.58%)
HPC_Pout_st        2147764.6699    2313704.0600    

In [16]:
# Test with CLIMB2 context
print("\n\n")
result_climb2 = check_simulator_consistency(
    df, 
    row_index=0, 
    context_name="CLIMB2", 
    sensor_names=["HPC_Tin", "LPT_Tin", "HPC_Pout_st"]
)




Row 0, Context: CLIMB2

STEP 1: State vector from CSV
  [0] deg_CmpBst_s_mapEff_in: -0.000612
  [1] deg_CmpBst_s_mapWc_in: -0.000174
  [2] deg_CmpFan_s_mapEff_in: 0.000031
  [3] deg_CmpFan_s_mapWc_in: -0.000013
  [4] deg_CmpH_s_mapEff_in: -0.001943
  [5] deg_CmpH_s_mapWc_in: 0.000043
  [6] deg_TrbH_s_mapEff_in: -0.001463
  [7] deg_TrbH_s_mapWc_in: 0.000065
  [8] deg_TrbL_s_mapEff_in: -0.000555
  [9] deg_TrbL_s_mapWc_in: -0.000015

STEP 2: Flight conditions from CSV
  DTAMB: 10.784
  ALT: 35044.452249
  MACH: 0.783448
  COMMAND: 24981.32783

STEP 3: Created context with PHASE_TYPE=MCL

STEP 4: Simulator called successfully

STEP 5: COMPARISON
Sensor                CSV Value       Simulated            Diff     Match?
----------------------------------------------------------------------
HPC_Tin                358.6787        338.0174        -20.6614  ✗ (5.76%)
LPT_Tin               1141.6611       1143.8394          2.1783  ✗ (0.19%)
HPC_Pout_st        1392965.9960    1419472.0800    

In [19]:
# Maybe the CSV was generated with DEFAULT flight conditions, not the ones in the CSV?
def check_with_default_conditions(df, row_index, context_name, sensor_names):
    """
    Check if CSV values match when using DEFAULT (template) flight conditions
    instead of the flight conditions from the CSV.
    """
    row = df.iloc[row_index]
    state = np.array([row[label] for label in STATE_LABELS])
    
    # Use DEFAULT context (not from CSV)
    context = CONTEXT_TEMPLATE[context_name]
    
    sensors_list = [SENSOR_OBJECTS[s] for s in sensor_names]
    result = decksmr_1forall([state], [context], sensors_list, ROOT_OPENDECK)
    
    print(f"\nUsing DEFAULT flight conditions for {context_name}:")
    fc = context.flight_condition
    print(f"  DTAMB={fc.DTAMB}, ALT={fc.ALT}, MACH={fc.MACH}, COMMAND={fc.COMMAND}")
    
    print(f"\n{'Sensor':<15} {'CSV Value':>15} {'Sim (Default)':>15} {'Diff':>15}")
    print("-"*60)
    
    for sensor_name in sensor_names:
        csv_col = f"{context_name}_DECKSMR{sensor_name}"
        csv_value = row[csv_col]
        sim_value = result[sensor_name].values[0]
        diff = sim_value - csv_value
        print(f"{sensor_name:<15} {csv_value:>15.4f} {sim_value:>15.4f} {diff:>15.4f}")

In [20]:
# Test with default conditions
print("COMPARING: CSV flight conditions vs DEFAULT flight conditions")
print("="*70)

print("\n--- Using CSV flight conditions ---")
_ = check_simulator_consistency(df, 0, "CRUISE", ["HPC_Tin", "LPT_Tin", "HPC_Pout_st"])

print("\n--- Using DEFAULT flight conditions ---")
check_with_default_conditions(df, 0, "CRUISE", ["HPC_Tin", "LPT_Tin", "HPC_Pout_st"])

COMPARING: CSV flight conditions vs DEFAULT flight conditions

--- Using CSV flight conditions ---
Row 0, Context: CRUISE

STEP 1: State vector from CSV
  [0] deg_CmpBst_s_mapEff_in: -0.000612
  [1] deg_CmpBst_s_mapWc_in: -0.000174
  [2] deg_CmpFan_s_mapEff_in: 0.000031
  [3] deg_CmpFan_s_mapWc_in: -0.000013
  [4] deg_CmpH_s_mapEff_in: -0.001943
  [5] deg_CmpH_s_mapWc_in: 0.000043
  [6] deg_TrbH_s_mapEff_in: -0.001463
  [7] deg_TrbH_s_mapWc_in: 0.000065
  [8] deg_TrbL_s_mapEff_in: -0.000555
  [9] deg_TrbL_s_mapWc_in: -0.000015

STEP 2: Flight conditions from CSV
  DTAMB: 9.422
  ALT: 35000.102656
  MACH: 0.780833
  COMMAND: 24961.358645

STEP 3: Created context with PHASE_TYPE=CR

STEP 4: Simulator called successfully

STEP 5: COMPARISON
Sensor                CSV Value       Simulated            Diff     Match?
----------------------------------------------------------------------
HPC_Tin                356.0334        335.2318        -20.8016  ✗ (5.84%)
LPT_Tin               1132.4634

---
## Debug: Check State Vector Ordering

Maybe the simulator expects a different ordering of the state vector?

In [22]:
# Print the exact STATE_LABELS ordering
print("STATE_LABELS ordering (this is what the simulator expects):")
for i, label in enumerate(STATE_LABELS):
    print(f"  state[{i}] = {label}")

print("\nValues from row 0:")
row = df.iloc[0]
for i, label in enumerate(STATE_LABELS):
    print(f"  state[{i}] = {label} = {row[label]:.6f}")

STATE_LABELS ordering (this is what the simulator expects):
  state[0] = deg_CmpBst_s_mapEff_in
  state[1] = deg_CmpBst_s_mapWc_in
  state[2] = deg_CmpFan_s_mapEff_in
  state[3] = deg_CmpFan_s_mapWc_in
  state[4] = deg_CmpH_s_mapEff_in
  state[5] = deg_CmpH_s_mapWc_in
  state[6] = deg_TrbH_s_mapEff_in
  state[7] = deg_TrbH_s_mapWc_in
  state[8] = deg_TrbL_s_mapEff_in
  state[9] = deg_TrbL_s_mapWc_in

Values from row 0:
  state[0] = deg_CmpBst_s_mapEff_in = -0.000612
  state[1] = deg_CmpBst_s_mapWc_in = -0.000174
  state[2] = deg_CmpFan_s_mapEff_in = 0.000031
  state[3] = deg_CmpFan_s_mapWc_in = -0.000013
  state[4] = deg_CmpH_s_mapEff_in = -0.001943
  state[5] = deg_CmpH_s_mapWc_in = 0.000043
  state[6] = deg_TrbH_s_mapEff_in = -0.001463
  state[7] = deg_TrbH_s_mapWc_in = 0.000065
  state[8] = deg_TrbL_s_mapEff_in = -0.000555
  state[9] = deg_TrbL_s_mapWc_in = -0.000015


In [ ]:
STATE_LABELS